# **Initial Phase(by Elsa) **


*   Importation of libraries and Json file
*   Overview of the raw dataset
*   Data cleaning
*   Data preparation





In [ ]:
# Cell 1: Download data from API and save it locally (Run this only ONCE)
import requests
import pandas as pd
import os

# Define the local file name
csv_file = "eco2mix.csv"

# Check if the file already exists to avoid re-downloading
if not os.path.exists(csv_file):
    print(f"Local file '{csv_file}' not found. Downloading from the API...")
    
    # GET request on the API endpoint
    try:
        response = requests.get('https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/eco2mix-regional-cons-def/exports/json?lang=fr&timezone=Europe%2FBerlin' )
        response.raise_for_status() # Raise an exception for bad status codes
        response_json = response.json()

        # Store content in a DataFrame
        df_energy = pd.DataFrame(response_json)

        # Save the dataframe to a local CSV file
        df_energy.to_csv(csv_file, index=False)

        print(f"Data downloaded and saved to '{csv_file}' successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading data: {e}")
else:
    print(f"Local file '{csv_file}' already exists. Skipping download.")



In [ ]:
# Cell 2: Load data from the local CSV and start processing (Start your work from here every day)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

# Load the data from the local file
df_energy = pd.read_csv("eco2mix.csv")

# --- Elsa's original cleaning code starts here ---
# (The code for displaying shape, dtypes, head, etc.)

#general info on the dataframe
print("Shape of the initial dataframe:", df_energy.shape)
print("\nData types of columns:")
display(df_energy.dtypes)

# overview of the dataset
print("\nFirst 10 rows of the dataset:")
df_energy.head(10)


In [ ]:
#data cleaning phase - first analyses

#start viewing the missing values
print("nb of columns of the transactions DataFrame containing missing values :",df_energy.isna().any(axis=0).sum(axis=0))
print('total missing values :')
display (df_energy.isna().sum())

#check if there are more missing values on aggregated data than definitive data
print('missing values for aggregated data :')
display (df_energy.loc[df_energy['nature']=='Données consolidées'].isna().sum())
print('missing values for definitive data :')
display (df_energy.loc[df_energy['nature']=='Données définitives'].isna().sum())

#As column_30 is empty on all rows, we can erase it from our cleaned dataframe (df_energy_cleaned)
df_energy_cleaned=df_energy[['code_insee_region','libelle_region','nature','date','heure','date_heure','consommation','thermique','nucleaire','eolien','solaire','hydraulique','pompage','bioenergies','ech_physiques', 'stockage_batterie', 'destockage_batterie', 'eolien_terrestre','eolien_offshore','tco_thermique','tch_thermique','tco_nucleaire','tch_nucleaire','tco_eolien','tch_eolien','tco_solaire','tch_solaire','tco_hydraulique','tch_hydraulique','tco_bioenergies','tch_bioenergies']]
print ("shape of the target dataframe=",df_energy_cleaned.shape)

#check negative values
cols = ['consommation','thermique','nucleaire','solaire','hydraulique','pompage','bioenergies','ech_physiques', 'stockage_batterie', 'destockage_batterie', 'eolien_terrestre','eolien_offshore','tco_thermique','tch_thermique','tco_nucleaire','tch_nucleaire','tco_eolien','tch_eolien','tco_solaire','tch_solaire','tco_hydraulique','tch_hydraulique','tco_bioenergies','tch_bioenergies']
for col in cols:
    print(f"negative values '{col}':",
          (df_energy_cleaned[col] < 0).sum())


In [ ]:
#data cleaning phase - Corrections of values and formats
#set all numerical missing values to 0
df_energy_cleaned=df_energy_cleaned.fillna(0)

#convert to the proper type of data columns date, heure and date_heure
df_energy_cleaned['date_heure']=df_energy_cleaned['date_heure'].astype('datetime64[ns, UTC]')
df_energy_cleaned['date']=pd.to_datetime( df_energy_cleaned['date'],format='ISO8601')
df_energy_cleaned['heure']== pd.to_datetime(df_energy_cleaned["heure"], format="%H:%M").dt.time

# remove anomalies from eolien
df_energy_cleaned=df_energy_cleaned.replace({'eolien': 'nan'}, 0)
df_energy_cleaned=df_energy_cleaned.replace({'eolien': 'ND'}, 0)
df_energy_cleaned=df_energy_cleaned.replace({'eolien': '-'}, 0)
df_energy_cleaned['eolien'] = df_energy_cleaned['eolien'].astype ("Float64")
#print("negative values 'eolien':",df_energy_cleaned['eolien'].loc[df_energy_cleaned['eolien']<0].count())
#print('')
df_energy_cleaned.loc[df_energy_cleaned['eolien'] < 0, 'eolien'] = 0

# replace negative values by 0 for columns 'thermique',Nucleaire, 'solaire','hydraulique','pompage', 'ech_physiques'
df_energy_cleaned.loc[df_energy_cleaned['thermique'] < 0, 'thermique'] = 0
cols = ['solaire', 'hydraulique','pompage', 'ech_physiques','nucleaire']
df_energy_cleaned[cols] = df_energy_cleaned[cols].clip(lower=0)
#for col in cols:
    #print(f"negative values '{col}':",
          #(df_energy_cleaned[col] < 0).sum())

print('Cleaning phase is finished.')


In [ ]:
#Data preparation  - check data quality

#list values of qualitative attributes
display (df_energy_cleaned['code_insee_region'].unique())
display (df_energy_cleaned['libelle_region'].unique())
display (df_energy_cleaned['nature'].unique())

#check distribution of quantitative attributes
display(df_energy_cleaned.describe())

Related Data Audit file is available here :

```
# https://docs.google.com/spreadsheets/d/1Sm4n9Jx5DVdKHfUIZrq4sKB0bMYWmoF6naFtgWgiXU4/edit?usp=sharing
```



In [ ]:
df_energy_cleaned.head(20)


In [ ]:
df_energy_cleaned.to_csv("df_energy_cleaned", index=False)

# **Analysis phase (by Pascal)**

In [ ]:
# REGIONAL STATISTICAL ANALYSIS
# Group the dataset by region
# dataset has a panel structure (region x time)
# compute statistics per region instead of global aggregates


group_cols = ["code_insee_region", "libelle_region"]

# I select the main quantitative energy variables.
# These represent production and consumption levels in MW.
energy_cols = [
    "consommation",
    "thermique",
    "nucleaire",
    "eolien",
    "solaire",
    "hydraulique",
    "pompage",
    "bioenergies",
    "ech_physiques"
]

# Keep only variables that actually exist in the dataset
energy_cols = [col for col in energy_cols if col in df_energy_cleaned.columns]

print("Energy variables used for analysis:", energy_cols)

# Non-convertible values will be transformed into NaN.

df_tmp = df_energy_cleaned.copy()

for col in energy_cols:
    df_tmp[col] = pd.to_numeric(df_tmp[col], errors="coerce")

# Region codes should also be numeric for proper sorting
df_tmp["code_insee_region"] = pd.to_numeric(
    df_tmp["code_insee_region"], errors="coerce"
)

# I aggregate the data by region and compute:
# - mean → average production/consumption level
# - std  → volatility over time
# - min  → lowest observed value
# - max  → peak value

regional_stats = (
    df_tmp
    .groupby(group_cols)[energy_cols]
    .agg(["mean", "std", "min", "max"])
)

# Flatten the multi-level column names for readability
regional_stats.columns = [
    f"{variable}_{stat}" for variable, stat in regional_stats.columns
]

# Convert index back to regular columns and sort by region code
regional_stats = (
    regional_stats
    .reset_index()
    .sort_values("code_insee_region")
)

# Display final table
display(regional_stats)


The regional statistics reveal substantial heterogeneity in electricity consumption across French regions.

Île-de-France and Auvergne-Rhône-Alpes exhibit the highest average consumption levels, reflecting their economic and demographic importance.

Smaller regions such as Centre-Val de Loire and Bretagne display significantly lower average demand, consistent with lower population density.

The standard deviation of consumption scales with region size, indicating that larger regions experience stronger absolute fluctuations.

However, volatility should ideally be evaluated relative to the mean (coefficient of variation), as absolute standard deviation alone may be misleading.

Thermal production shows strong regional concentration, particularly in Grand Est and Hauts-de-France, suggesting structural differences in generation infrastructure.

Nuclear production is unevenly distributed, reflecting the geographic placement of nuclear power plants.

Wind production (eolien) demonstrates higher volatility compared to nuclear generation, highlighting the intermittency of renewable sources.

Solar production remains relatively modest in absolute terms but likely exhibits strong intra-day and seasonal variability.

Hydroelectric production appears more stable but varies depending on regional natural resources and reservoir capacity.

Several regions show significant negative values in physical exchanges (ech_physiques), indicating net electricity exports.

In contrast, Île-de-France exhibits large positive exchange values, suggesting structural dependence on imports from other regions.

Pumped storage (pompage) activity is limited to specific regions, indicating localized storage infrastructure.

The energy mix differs substantially across regions, underlining the importance of region-level analysis rather than global aggregation.

Overall, the dataset confirms that regional electricity systems differ in scale, volatility, and structural composition, which has implications for grid stability and energy policy analysis.

# **Initial Data Exploration (by Ahmed)**

In [ ]:
#
# --- Final Corrected Code - Version 4 ---

# 1. Make a fresh, safe copy of the cleaned data
data = df_energy_cleaned.copy()

# 2. Correct the 'eolien' column
data['eolien'] = data['eolien'].astype(str).str.replace(',', '.', regex=False)
data['eolien'] = pd.to_numeric(data['eolien'], errors='coerce')

# 3. Aggregate Consumption at the national level
# We assume all rows can potentially describe consumption
national_consumption = data.groupby('date_heure')['consommation'].sum().reset_index()

# 4. Aggregate Production at the national level
# We assume the SAME rows can also describe production
production_cols = ['thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']
data['total_production'] = data[production_cols].fillna(0).sum(axis=1)
national_production = data.groupby('date_heure')['total_production'].sum().reset_index()

# 5. Merge the national data
df_national = pd.merge(national_consumption, national_production, on='date_heure')

# --- Final Inspection ---
print("--- Final National DataFrame ---")
print("Shape:", df_national.shape)
print("\nFirst 5 rows:")
display(df_national.head())
print("\nStatistics:")
display(df_national.describe())

In [ ]:
# --- Figure 1 (FIXED): National Consumption vs. Production ---

# 1. Aggregate data to a national, daily level
df_national_temp = df_energy_cleaned.copy()
df_national_temp['date'] = pd.to_datetime(df_national_temp['date_heure']).dt.date
production_cols = ['thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']
df_national_temp['total_production'] = df_national_temp[production_cols].sum(axis=1)
df_national_simple = df_national_temp.groupby('date')[['consommation', 'total_production']].mean().reset_index()

# 2. Create the plot
fig1 = go.Figure()

# 3. Add the traces using a more stable shading method
# First, draw the consumption line (bottom line)
fig1.add_trace(go.Scatter(
    x=df_national_simple['date'], y=df_national_simple['consommation'],
    mode='lines', name='National Consumption', line=dict(color='royalblue', width=2)
))
# Second, draw the production line (top line) and tell it to fill the area down to the previous trace
fig1.add_trace(go.Scatter(
    x=df_national_simple['date'], y=df_national_simple['total_production'],
    mode='lines', name='National Production', line=dict(color='green', width=2),
    fill='tonexty', fillcolor='rgba(0,100,80,0.2)' # Fill down to the next Y trace (which is consumption)
))

# 4. Update layout
fig1.update_layout(
    title_text='National Energy Balance: Production vs. Consumption',
    xaxis_title='Date', yaxis_title='Average Daily Energy (MW)',
    xaxis_rangeslider_visible=True
)
fig1.show()


Figure 1: National Energy Consumption vs. Production (2013-2024) . This chart illustrates the national energy balance over the last decade. A clear seasonal pattern is visible, with both consumption and production peaking during the winter months. Notably, national production consistently exceeds consumption, indicating a surplus that is likely exported.

4. Regional Level Analysis: Consumption vs. Production (By Ahmed)

In [ ]:
#
# --- Prepare Data for Regional Analysis ---

# We will use the 'data' DataFrame which is already cleaned.
# We just need to group it by region AND timestamp.

# Group by region and date to get consumption for each region over time.
regional_consumption = data.groupby(['libelle_region', 'date_heure'])['consommation'].sum().reset_index()

# Group by region and date to get total production for each region over time.
regional_production = data.groupby(['libelle_region', 'date_heure'])['total_production'].sum().reset_index()

# Merge the two dataframes to have everything in one place.
df_regional = pd.merge(regional_consumption, regional_production, on=['libelle_region', 'date_heure'])

print("Regional data prepared successfully.")
display(df_regional.head())

In [ ]:
# --- Figure 2: Interactive Analysis of Regional Energy Balance ---
# (Comparing regional production to the national average)

df_regional_temp = df_energy_cleaned.copy()
df_regional_temp['date'] = pd.to_datetime(df_regional_temp['date_heure']).dt.date
production_cols = ['thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']
df_regional_temp['total_production'] = df_regional_temp[production_cols].sum(axis=1)
df_daily_regional = df_regional_temp.groupby(['date', 'libelle_region'])['total_production'].mean().reset_index()
df_national_avg = df_regional_temp.groupby('date')['total_production'].mean().reset_index()
df_national_avg.rename(columns={'total_production': 'national_avg_production'}, inplace=True)

list_of_regions = sorted(df_daily_regional['libelle_region'].unique())
first_region = list_of_regions[0]
df_filtered = df_daily_regional[df_daily_regional['libelle_region'] == first_region]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df_filtered['date'], y=df_filtered['total_production'], name='Regional Production', line_color='royalblue'))
fig2.add_trace(go.Scatter(x=df_national_avg['date'], y=df_national_avg['national_avg_production'], name='National Average', line=dict(color='black', width=2, dash='dash')))

buttons = []
for region in list_of_regions:
    df_button_filtered = df_daily_regional[df_daily_regional['libelle_region'] == region]
    buttons.append(dict(label=region, method='restyle', args=[{'x': [df_button_filtered['date']], 'y': [df_button_filtered['total_production']]}, [0]]))

fig2.update_layout(
    title_text=f'Regional Production vs. National Average: {first_region}',
    xaxis_title='Date', yaxis_title='Average Daily Production (MW)',
    updatemenus=[dict(active=0, buttons=buttons, direction="down", x=0.1, xanchor="left", y=1.15, yanchor="top")]
)
fig2.show()


Figure 2: Interactive Analysis of Regional Energy Balance. This interactive tool allows for a detailed examination of the energy balance within each French region. By selecting a region from the dropdown menu, we can observe specific local patterns. For example, regions with high industrial activity show different consumption profiles compared to more residential areas."""

"Analysis by production sector: nuclear/renewable energy"

*   Élément de liste

*   Élément de liste
*   Élément de liste


In [ ]:
# --- Figure 3 (FIXED): Composition of National Energy Production ---

# 1. Aggregate data to a national, daily level
df_nat_detailed_temp = df_energy_cleaned.copy()
df_nat_detailed_temp['date'] = pd.to_datetime(df_nat_detailed_temp['date_heure']).dt.date
energy_cols = ['thermique', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 'bioenergies']
df_national_detailed = df_nat_detailed_temp.groupby('date')[energy_cols].mean().reset_index()

# --- NEW: Calculate Total Production to replace Consumption ---
df_national_detailed['total_production'] = df_national_detailed[energy_cols].sum(axis=1)

# 2. Create the plot
fig3 = go.Figure()

# 3. Add stacked traces for non-nuclear production
# (Code for adding stacked traces remains the same)
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['hydraulique'], name='Hydro', stackgroup='one', line_width=0, fillcolor='dodgerblue'))
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['eolien'], name='Wind', stackgroup='one', line_width=0, fillcolor='lightgreen'))
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['solaire'], name='Solar', stackgroup='one', line_width=0, fillcolor='gold'))
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['thermique'], name='Thermal', stackgroup='one', line_width=0, fillcolor='darkred'))
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['bioenergies'], name='Bioenergy', stackgroup='one', line_width=0, fillcolor='saddlebrown'))

# 4. Add separate lines for Nuclear and TOTAL PRODUCTION
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['nucleaire'], name='Nuclear', line=dict(color='orange', width=2.5)))
# --- THIS IS THE CORRECTED LINE ---
fig3.add_trace(go.Scatter(x=df_national_detailed['date'], y=df_national_detailed['total_production'], name='Total Production', line=dict(color='black', width=3, dash='dash')))

# 5. Update layout
fig3.update_layout(
    title_text='Composition of National Energy Production by Sector',
    xaxis_title='Date', yaxis_title='Average Daily Energy (MW)',
    xaxis_rangeslider_visible=True
)
fig3.show()


Figure 3: Composition of National Energy Production by Sector.                                                           The stacked area chart reveals the composition of France's energy mix. Nuclear power (orange) forms the vast and stable baseload of production. We can also observe the seasonal contribution of hydraulic power (blue) and the smaller but growing shares of solar (gold) and wind (green) energy over the years.

In [ ]:
# --- Figure 4 (FIXED & FINAL): Geographical Distribution of Renewable Energy ---

# --- NEW: Import plotly.express ---
import plotly.express as px
import plotly.graph_objects as go
import requests # Make sure requests is imported if not done in a previous cell

# 1. Prepare data
renewable_cols = ['eolien', 'solaire', 'hydraulique', 'bioenergies']
# Ensure df_energy_cleaned exists from previous cells
renewable_by_region = df_energy_cleaned.groupby('libelle_region')[renewable_cols].sum().reset_index()
renewable_by_region['total_renewable'] = renewable_by_region[renewable_cols].sum(axis=1)

# 2. Load GeoJSON
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/regions.geojson"
try:
    response = requests.get(geojson_url )
    geojson_regions = response.json()
except Exception as e:
    print(f"Failed to load GeoJSON file: {e}")
    geojson_regions = None

# 3. Create the Interactive Map
if geojson_regions:
    fig4_map = go.Figure()
    
    # English names for the dropdown
    columns_to_plot = {'Total Renewable': 'total_renewable', 'Hydro': 'hydraulique', 'Wind': 'eolien', 'Solar': 'solaire', 'Bioenergy': 'bioenergies'}

    # Create a trace for each energy type
    for display_name, actual_col in columns_to_plot.items():
        fig4_map.add_trace(go.Choropleth(
            geojson=geojson_regions, locations=renewable_by_region['libelle_region'],
            featureidkey="properties.nom", z=renewable_by_region[actual_col],
            colorscale="Viridis", colorbar_title="GWh", name=display_name, visible=(display_name == 'Total Renewable') # Make first one visible
        ))

    # Create dropdown buttons
    buttons = []
    for i, display_name in enumerate(columns_to_plot.keys()):
        visibility = [False] * len(columns_to_plot)
        visibility[i] = True
        buttons.append(dict(label=display_name, method="update", args=[{"visible": visibility}]))

    fig4_map.update_layout(
        title_text='Geographical Distribution of Renewable Energy Production', title_x=0.5,
        updatemenus=[dict(active=0, buttons=buttons, direction="down", x=0.05, xanchor="left", y=0.95, yanchor="top")],
        geo=dict(scope='europe', fitbounds="locations", visible=False)
    )
    print("--- Interactive Map ---")
    fig4_map.show()

# 4. --- Create the Bar Chart for Top Renewable Regions ---
print("\n--- Top 10 Renewable Energy Producing Regions (Bar Chart) ---")
top_10_regions = renewable_by_region.sort_values('total_renewable', ascending=False).head(10)

# Use the 'px' alias which is now defined
fig4_bar = px.bar(
    top_10_regions,
    x='libelle_region',
    y='total_renewable',
    title='Top 10 Regions by Total Renewable Production',
    labels={'libelle_region': 'Region', 'total_renewable': 'Total Production (GWh)'},
    color='total_renewable',
    color_continuous_scale='Viridis'
)
fig4_bar.show()


Figure 4: Geographical Distribution of Total Renewable Energy Production. This choropleth map answers the question of "where" renewable energy is located. It highlights that production is not evenly distributed. The Auvergne-Rhône-Alpes region is the dominant producer, largely due to its significant hydraulic resources. Other regions like Occitanie also show substantial renewable output, indicating strong wind and solar capacities.

## Data Integrity Analysis - Missing and Negative Values (by Anne-Sophie)

In [ ]:
missing_percentage = (df_energy.isna().mean() * 100).sort_values(ascending= False)
percent_df = missing_percentage.reset_index()
percent_df.columns = ['Column Name', 'Missing Values']
percent_df

## Missing Values - Interpretation
The missing values are not uniformly distributed across variables.

column_30 is entirely empty (100%) and was removed during preprocessing. Several technical indicators (tch_*, tco_*) and some specific technologies (e.g. offshore wind, battery storage) show more than 60% missing values.
Several technical indicators (tch_*, tco_*) and some specific technologies (e.g. offshore wind, battery storage) show more than 60% missing values.
Core variables such as consommation, thermique, hydraulique, and solaire have almost no missing values and are therefore reliable for time-series analysis and forecasting.

In [ ]:
df_energy.shape

In [ ]:
df_energy[df_energy["nucleaire"].isna()]["date_heure"].describe()

In [ ]:
df_energy.loc[df_energy["nucleaire"].notna(), "date_heure"].min()

In [ ]:
df_energy.groupby("libelle_region")["nucleaire"].apply(lambda x: x.isna().mean()).sort_values(ascending=False)

Nuclear Production – Missing Pattern
Although nucleaire contains about 27% missing values, non-null values are present from the beginning of the dataset (2013), indicating that the variable was not introduced later.

The missing values are strongly region-dependent: some regions show almost no missing values, while others have around 66%.

This suggests that the missing values reflect structural regional differences (absence of nuclear plants) rather than data quality issues.

In [ ]:
df_energy[df_energy["libelle_region"] == "Bretagne"]["nucleaire"].describe()

In [ ]:
region = "Bretagne"

subset = df_energy[df_energy["libelle_region"] == region]

nb_nan = subset["nucleaire"].isna().sum()
nb_zero = (subset["nucleaire"] == 0).sum()
nb_total = len(subset)

nb_nan, nb_zero, nb_total

In [ ]:
subset[subset["nucleaire"].isna()]["date_heure"].min()

In [ ]:
subset[subset["nucleaire"].isna()]["date_heure"].max()

In [ ]:
subset[subset["nucleaire"].notna()]["date_heure"].min()

## Nuclear production – Case study: Bretagne
In Bretagne, we observed a mix of missing values and zeros for the nucleaire variable.

To understand this pattern, we examined the temporal distribution of missing values. We found that nuclear production is missing from 2013-01-01 to 2020-12-31.

From 2021-01-01 onwards, the variable is explicitly reported and equals 0.

This suggests that before 2021, nuclear production was not regionally reported for Bretagne. From 2021 onwards, the reporting system was updated and zero production is explicitly recorded for regions without nuclear plants.

In [ ]:
df_energy[df_energy["nucleaire"].notna()].groupby("libelle_region")["date_heure"].min()

## Conclusion on Nuclear Missing Values
The missing values in nucleaire are not random.

They reflect a structural change in regional reporting around 2021.
Before 2021, some regions did not report nuclear production at the regional level.
From 2021 onwards, zero production is explicitly recorded.

These missing values therefore correspond to reporting structure rather than data quality issues.

## Additional variable check to confirm the missing-value pattern
To verify whether the observed transition from missing values to zeros reflects a broader reporting change, we apply the same temporal analysis to another technology variable (eolien_offshore) and examine whether a comparable pattern emerges across regions.

In [ ]:
df_energy[df_energy["eolien_offshore"].notna()] \
    .groupby("libelle_region")["date_heure"] \
    .min() \
    .sort_values()

In [ ]:
region = 'Bretagne'
subset = df_energy[df_energy['libelle_region'] == region]

subset[['eolien_offshore']].describe()

In [ ]:
subset[subset['eolien_offshore'].isna()]['date_heure'].min(), \
subset[subset['eolien_offshore'].isna()]['date_heure'].max()

In [ ]:
subset_post = subset[subset['date_heure'] >= '2021-01-01']
(subset_post['eolien_offshore'] == 0).mean()

The analysis of eolien_offshore shows a pattern similar to what was observed for nuclear production. Values are missing until early 2021, after which the variable is consistently reported and equals zero for regions without offshore generation. This confirms that missing values are mainly driven by reporting practices rather than underlying data quality issues.

## Analysis of Negative Values in Nuclear Production

In [ ]:
# Count of the total number of negative nuclear production values
(df_energy["nucleaire"] < 0).sum()

In [ ]:
# Calculation of the percentage of negative nuclear values in the dataset
(df_energy["nucleaire"] < 0).mean() * 100

In [ ]:
# Descriptive Statistics for negative nuclear values
df_energy[df_energy["nucleaire"] < 0]["nucleaire"].describe()

In [ ]:
# Distribution of the negative nuclear values across the regions
df_energy[df_energy["nucleaire"] < 0]["libelle_region"].value_counts()

Negative nuclear values are exclusively observed in the Occitanie region.

In [ ]:
df_energy[df_energy["nucleaire"] < 0]["date_heure"].min(), \
df_energy[df_energy["nucleaire"] < 0]["date_heure"].max()

Negative nuclear values are rare (0.018%), limited to Occitanie, and occur between 2013 and 2019. Their magnitude is small compared to average nuclear production, suggesting reporting adjustments rather than actual negative production. We could decide to replace negative nuclear values with 0, as nuclear production cannot be negative.

## Analysis of Negative Values in Thermal Production
To determine whether negative values in thermal production correspond to localized reporting adjustments or a broader data issue, we examine their regional distribution across the dataset.

In [ ]:
df_energy[df_energy['thermique'] < 0]['libelle_region'].value_counts()

In [ ]:
(df_energy['thermique'] < 0).mean() *100

In [ ]:
df_energy[df_energy['thermique'] < 0]['thermique'].describe()

In [ ]:
df_energy[df_energy['thermique'] < 0]['date_heure'].describe()

In [ ]:
df_energy[df_energy['thermique'] < 0]['date_heure'].min(), \
df_energy[df_energy['thermique'] < 0]['date_heure'].max()

Negative values in thermal production are observed across several regions, with a strong concentration in Île-de-France and Occitanie. They occur between 2013 and late 2019 and are generally small in magnitude, suggesting adjustment or balancing effects rather than actual negative generation. The disappearance of these values after 2019 indicates a change in reporting or calculation practices rather than a persistent data quality issue.

## Monthly Gap Between Energy Production and Consumption (Yearly Comparison)

In [ ]:
# Extraction of year and month
df_national['year'] = df_national['date_heure'].dt.year
df_national['month'] = df_national['date_heure'].dt.month

# Compute monthly average consumption and production
monthly_avg = (
    df_national
    .groupby(['year', 'month'])[['consommation', 'total_production']]
    .mean()
    .reset_index()
)

# Calculate monthly production-consumption gab
monthly_avg['gap'] = monthly_avg['total_production'] - monthly_avg['consommation']

# Map month numbers to readable labels
month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

monthly_avg['month_name'] = monthly_avg['month'].map(month_map)

# Prepare ordered list of months for dropdown
list_of_months = monthly_avg['month_name'].unique()
ordered_months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                  'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
list_of_months = [m for m in ordered_months if m in set(monthly_avg['month_name'])]

monthly_avg.head()

In [ ]:
# Interactive chart: yearly evolution for a selected month
fig5 = go.Figure()

# Creation of 2 traces per month (production + consumption with area fill)
for month in list_of_months:
    df_month = monthly_avg[monthly_avg['month_name'] == month]

    fig5.add_trace(go.Scatter(
        x=df_month['year'],
        y=df_month['gap'],
        name=month,
        visible=False
    ))

# Default view: show the first month
fig5.data[0].visible = True

# Build dropdown buttons
month_buttons = []

for i, month in enumerate(list_of_months):
    month_visibility = [False] * len(list_of_months)
    month_visibility[i] = True

    month_button = dict(
        label=month, 
        method = 'update',
        args=[{'visible': month_visibility},
              {'title': f'Monthly Gap Evolution - {month}'}]
    )
    month_buttons.append(month_button)

# Layout
fig5.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=month_buttons,
        direction='down',
        showactive=True,
        x=0.1,
        xanchor='left',
        y=1.15,
        yanchor='top'
    )],
    title=f'Monthly Gap Evolution Production vs. Consumption - {list_of_months[0]}',
    xaxis_title='Year',
    yaxis_title='Production - Consumption (MW)'
)

fig5.show()

Figure 5: Monthly Comparison of National Energy Production and Consumption (2013–2024). This interactive chart presents the yearly evolution of average production and consumption for a selected calendar month, enabling direct comparison of winter demand and supply conditions across years. The shaded area highlights the gap between production and consumption, making periods of surplus or near-balance immediately visible. While production generally exceeds consumption over the period, temporary convergences and reversals appear in specific years, reflecting climatic variability, demand fluctuations and structural changes in generation availability.

In [ ]:
df_energy_cleaned.head(10)
df_energy_cleaned.shape


Time Handling and Feature Engineering
Objective by Pascal 

This section prepares and standardizes the datetime information of the dataset to enable reliable time-based analyses. The main goals are:

Convert the date_heure column (string format) into a proper datetime object with a consistent timezone (UTC).

Create an additional local time representation (Europe/Paris) to ensure correct hour-based comparisons (e.g., summer vs. winter, daily load profiles).

Generate meaningful time-based features (year, month, weekday, hour, season).

Prepare the dataset for time aggregations and grouping operations (daily, monthly, hourly analysis).

In [ ]:
# ============================================================
# TIME PREPROCESSING AND FEATURE ENGINEERING
# ============================================================

# Objective:
# This code standardizes and prepares the datetime column for
# reliable time-based analysis.

# Main goals:
# 1) Convert the raw timestamp (string format) into a proper
#    timezone-aware datetime object using a unified time basis (UTC).
#
# 2) Create an additional local time representation (Europe/Paris)
#    to correctly analyze hour-based patterns (e.g., daily load profiles,
#    summer vs. winter comparisons, peak hours).
#
# 3) Generate meaningful time-related features such as:
#    - year
#    - month
#    - weekday
#    - hour
#    - weekend indicator
#    - season classification (winter/summer)
#
# 4) Prepare the dataset for time-based aggregations such as:
#    - hourly profiles
#    - daily averages
#    - monthly trends
#    - seasonal comparisons
#
# Why this is important:
# Energy datasets often contain mixed time offsets due to daylight
# saving time (+01:00 / +02:00). Converting everything to UTC ensures
# a stable reference for resampling and long-term analysis, while
# local time allows correct interpretation of human-relevant patterns.
# ============================================================


#============= Code by Pascal ====================================
# ============================================================
# TIME HANDLING & FEATURE ENGINEERING
# ============================================================
# Goal:
# 1) Convert "date_heure" (string) into a proper datetime object
#    with a UNIFIED timezone (UTC).
# 2) Create an additional local time column (Europe/Paris)
#    to allow correct hour-based comparisons (summer vs winter,
#    daily load profiles, etc.).
# 3) Generate meaningful time features and prepare the dataset
#    for time-based grouping and resampling.
# ============================================================


# ============================================================
# TIME HANDLING & FEATURE ENGINEERING
# ============================================================
# Goal:
# 1) Convert "date_heure" (string) into a proper datetime object
#    with a UNIFIED timezone (UTC).
# 2) Create an additional local time column (Europe/Paris)
#    to allow correct hour-based comparisons (summer vs winter,
#    daily load profiles, etc.).
# 3) Generate meaningful time features and prepare the dataset
#    for time-based grouping and resampling.
# ============================================================


# ----------------------------
# 0) Define datetime column
# ----------------------------
datetime_col = "date_heure"

# Ensure the datetime column exists before proceeding.
# If the condition is False, the code stops immediately
# with an AssertionError.
assert datetime_col in df_energy_cleaned.columns, f"Column '{datetime_col}' not found!"


# ----------------------------
# 1) Convert string -> datetime (handle mixed timezones)
# ----------------------------
# Why utc=True?
# - The original strings contain timezone offsets (+01:00 / +02:00),
#   caused by daylight saving time (DST).
# - Without utc=True, pandas may raise:
#   "Mixed timezones detected".
# - utc=True converts everything into a UNIFIED time reference (UTC).
#
# errors="coerce":
# - Any invalid timestamp will be converted into NaT
#   (Not a Time), preventing the code from crashing.
df_energy_cleaned[datetime_col] = pd.to_datetime(
    df_energy_cleaned[datetime_col],
    utc=True,
    errors="coerce"
)

# Quick data quality check:
# Count how many timestamps could not be converted.
n_bad_dates = int(df_energy_cleaned[datetime_col].isna().sum())
print("Bad/NaT dates after conversion:", n_bad_dates)


# ----------------------------
# 2) Create local time column
# ----------------------------
# Why create a local time column?
# - UTC is ideal for stable resampling and long-term trends.
# - Local time is necessary for correct hour-based interpretation
#   (e.g., electricity demand at 18:00 local time).
#
# The dataset is French (eco2mix), therefore Europe/Paris is appropriate.
local_tz = "Europe/Paris"

df_energy_cleaned["date_local"] = df_energy_cleaned[datetime_col].dt.tz_convert(local_tz)


# ----------------------------
# 3) Generate time features (UTC + Local)
# ----------------------------

# ---- UTC-based features ----
# Useful for stable aggregation and resampling
df_energy_cleaned["year_utc"]  = df_energy_cleaned[datetime_col].dt.year
df_energy_cleaned["month_utc"] = df_energy_cleaned[datetime_col].dt.month
df_energy_cleaned["date_utc"]  = df_energy_cleaned[datetime_col].dt.date  # date without time


# ---- Local time-based features ----
# Important for daily patterns and behavioral analysis
df_energy_cleaned["year_local"]    = df_energy_cleaned["date_local"].dt.year
df_energy_cleaned["month_local"]   = df_energy_cleaned["date_local"].dt.month
df_energy_cleaned["day_local"]     = df_energy_cleaned["date_local"].dt.day
df_energy_cleaned["weekday_local"] = df_energy_cleaned["date_local"].dt.dayofweek  # 0=Mon ... 6=Sun
df_energy_cleaned["hour_local"]    = df_energy_cleaned["date_local"].dt.hour
df_energy_cleaned["minute_local"]  = df_energy_cleaned["date_local"].dt.minute

# Create a weekend flag (very useful in demand analysis)
df_energy_cleaned["is_weekend"] = df_energy_cleaned["weekday_local"].isin([5, 6])


# ----------------------------
# 4) Define seasons (Winter vs Summer)
# ----------------------------
# For seasonal comparisons, it is often cleaner to define
# seasons by months rather than relying on DST transitions.
#
# Example definition:
# - Winter: December, January, February
# - Summer: June, July, August
winter_months = [12, 1, 2]
summer_months = [6, 7, 8]

# np.select evaluates conditions row-wise:
# - If condition 1 is True -> assign "winter"
# - Else if condition 2 is True -> assign "summer"
# - Otherwise -> assign "other"
df_energy_cleaned["season"] = np.select(
    [
        df_energy_cleaned["month_local"].isin(winter_months),
        df_energy_cleaned["month_local"].isin(summer_months)
    ],
    [
        "winter",
        "summer"
    ],
    default="other"
)


# ----------------------------
# 5) Example aggregations
# ----------------------------
target_col = "consommation"
assert target_col in df_energy_cleaned.columns, f"Column '{target_col}' not found!"


# A) Average hourly profile (local time)
hourly_profile_all = (
    df_energy_cleaned
    .groupby("hour_local")[target_col]
    .mean()
    .sort_index()
)


# B) Hourly profile by season (winter vs summer)
hourly_profile_by_season = (
    df_energy_cleaned[df_energy_cleaned["season"].isin(["winter", "summer"])]
    .groupby(["season", "hour_local"])[target_col]
    .mean()
    .reset_index()
    .sort_values(["season", "hour_local"])
)


# C) Daily averages (UTC-based, stable resampling)
daily_mean_utc = (
    df_energy_cleaned
    .set_index(datetime_col)[target_col]
    .resample("D")
    .mean()
)


# D) Monthly averages (UTC-based)
monthly_mean_utc = (
    df_energy_cleaned
    .set_index(datetime_col)[target_col]
    .resample("ME")
    .mean()
)


# ----------------------------
# 6) Sanity checks
# ----------------------------
print("\n=== DATETIME TYPES ===")
print("date_heure dtype:", df_energy_cleaned[datetime_col].dtype)
print("date_local dtype:", df_energy_cleaned["date_local"].dtype)

print("\n=== DATE RANGE (UTC) ===")
print("min:", df_energy_cleaned[datetime_col].min())
print("max:", df_energy_cleaned[datetime_col].max())

print("\n=== HOURLY PROFILE (ALL) – first 5 hours ===")
print(hourly_profile_all.head())

print("\n=== HOURLY PROFILE BY SEASON – first rows ===")
print(hourly_profile_by_season.head())

print("\n=== DAILY MEAN (UTC) – first 5 days ===")
print(daily_mean_utc.head())

print("\n=== MONTHLY MEAN (UTC) – first 5 months ===")
print(monthly_mean_utc.head())


# ----------------------------
# 7) Show first rows with new time features
# ----------------------------
print("\n=== DATAFRAME HEAD (with new time features) ===")
print(df_energy_cleaned.head())


In [ ]:

display(df_energy_cleaned.head())



Daily Electricity Consumption Profile: Winter vs Summer

What This Visualization Shows:

This chart displays the average hourly electricity consumption within a selected region, comparing winter months (Dec–Feb) and summer months (Jun–Aug).

The x-axis represents the local hour of the day (0–23), while the y-axis shows the average electricity consumption.

Each line represents one season.

 User Options:

Region dropdown → Allows switching between individual regions.

The comparison automatically focuses on winter vs summer based on predefined month classifications.

 How to Interpret It:

This visualization helps identify:

Differences in peak demand levels

Changes in load curve shape

Seasonal shifts in morning and evening peaks

Typical findings may include:

Higher evening peaks during winter (heating, lighting demand)

Flatter consumption profiles during summer

Region-specific structural differences

If winter consumption is consistently above summer levels, this indicates strong seasonal demand dependence, often linked to heating requirements.

In [ ]:
# ============================================================
# WINTER VS SUMMER – DAILY CONSUMPTION PROFILE (INTERACTIVE)
# ============================================================

import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# ----------------------------
# Column definitions
# ----------------------------
region_col = "libelle_region"
cons_col = "consommation"

# Required time features (must already exist from your time preprocessing)
required_cols = ["hour_local", "month_local"]
missing = [c for c in required_cols if c not in df_energy_cleaned.columns]
assert not missing, f"Missing time feature columns: {missing}"

# Region list
regions = sorted(df_energy_cleaned[region_col].dropna().unique().tolist())

# ----------------------------
# Season definition
# ----------------------------
winter_months = [12, 1, 2]
summer_months = [6, 7, 8]


def plot_winter_vs_summer(region: str):
    # Filter region
    df = df_energy_cleaned[df_energy_cleaned[region_col] == region].copy()

    # Define clean two-season variable
    df["season_clean"] = np.select(
        [
            df["month_local"].isin(winter_months),
            df["month_local"].isin(summer_months)
        ],
        ["Winter", "Summer"],
        default="Other"
    )

    # Keep only winter and summer
    df = df[df["season_clean"].isin(["Winter", "Summer"])]

    # Aggregate: mean consumption by hour and season
    grouped = (
        df.groupby(["season_clean", "hour_local"])[cons_col]
        .mean()
        .reset_index()
        .sort_values(["season_clean", "hour_local"])
    )

    # Plot
    fig = px.line(
        grouped,
        x="hour_local",
        y=cons_col,
        color="season_clean",
        markers=True,
        title=f"Daily Consumption Profile: Winter vs Summer — {region}",
        template="plotly_white"
    )

    fig.update_layout(
        xaxis_title="Local Hour of Day",
        yaxis_title="Average Consumption",
        hovermode="x unified",
        legend_title="Season"
    )

    fig.update_xaxes(dtick=1)

    return fig


# ----------------------------
# Widgets
# ----------------------------
region_dropdown = widgets.Dropdown(
    options=regions,
    value=regions[0],
    description="Region:"
)

output = widgets.Output()


def update_plot(change=None):
    with output:
        output.clear_output(wait=True)
        display(plot_winter_vs_summer(region_dropdown.value))


region_dropdown.observe(update_plot, names="value")

display(region_dropdown, output)

update_plot()


Daily Electricity Consumption Profile: Weekday vs Weekend

 What This Visualization Shows:

This chart compares the average hourly electricity consumption between weekdays and weekends for a selected region.

The x-axis shows the hour of the day, and the y-axis represents average consumption.

Two lines represent:

Weekdays

Weekends

 User Options:

Region dropdown → Allows regional comparison.

 How to Interpret It:

This visualization reveals behavioral and economic patterns:

Lower industrial and commercial activity on weekends

Delayed morning peaks on weekends

Potential reduction in total daily demand

Key analytical insights include:

The magnitude of the weekday–weekend gap

Shifts in peak timing

Regional differences in economic structure

A strong divergence between weekday and weekend profiles often indicates a region with high industrial or commercial activity.

In [ ]:
# ============================================================
# WEEKDAY VS WEEKEND – DAILY CONSUMPTION PROFILE (INTERACTIVE)
# ============================================================
# ----------------------------
# Column definitions
# ----------------------------
region_col = "libelle_region"
cons_col = "consommation"

# Required time features (must already exist from your time preprocessing)
required_cols = ["hour_local", "is_weekend"]
missing = [c for c in required_cols if c not in df_energy_cleaned.columns]
assert not missing, f"Missing time feature columns: {missing}"

# Region list
regions = sorted(df_energy_cleaned[region_col].dropna().unique().tolist())


def plot_weekday_vs_weekend(region: str):
    # Filter region
    df = df_energy_cleaned[df_energy_cleaned[region_col] == region].copy()

    # Create readable label
    df["day_type"] = np.where(df["is_weekend"], "Weekend", "Weekday")

    # Aggregate: mean consumption by hour and day type
    grouped = (
        df.groupby(["day_type", "hour_local"])[cons_col]
        .mean()
        .reset_index()
        .sort_values(["day_type", "hour_local"])
    )

    # Plot
    fig = px.line(
        grouped,
        x="hour_local",
        y=cons_col,
        color="day_type",
        markers=True,
        title=f"Daily Consumption Profile: Weekday vs Weekend — {region}",
        template="plotly_white"
    )

    fig.update_layout(
        xaxis_title="Local Hour of Day",
        yaxis_title="Average Consumption",
        hovermode="x unified",
        legend_title="Day type"
    )

    fig.update_xaxes(dtick=1)

    return fig


# ----------------------------
# Widgets
# ----------------------------
region_dropdown = widgets.Dropdown(
    options=regions,
    value=regions[0],
    description="Region:"
)

output = widgets.Output()


def update_plot(change=None):
    with output:
        output.clear_output(wait=True)
        display(plot_weekday_vs_weekend(region_dropdown.value))


region_dropdown.observe(update_plot, names="value")

display(region_dropdown, output)

update_plot()

Energy Mix Evolution: 100% Stacked Production Shares

 What This Visualization Shows

This chart illustrates the composition of electricity production over time, expressed as percentage shares summing to 100%.

The y-axis shows share of total electricity production (%), while the x-axis represents time (monthly or yearly).

The production mix is divided into:

Renewables (aggregated or split by technology)

Nuclear

Thermal/Fossil

The total always equals 100%, highlighting structural shifts rather than volume changes.

User Options

Region dropdown → Select individual region or "France (all regions)"

Time aggregation dropdown → Monthly or yearly trend

Renewables split toggle → View renewables as:

Aggregated

Separated into Solar, Wind (onshore/offshore), Hydro, Bioenergy

 How to Interpret It

This visualization highlights structural transformation of the energy system:

Increasing renewable share → Progress in energy transition

Declining thermal share → Reduction of fossil dependency

Nuclear stability or decline → Policy or capacity shifts

Because the chart shows percentages rather than absolute values, it isolates structural change in the energy mix.

If renewables steadily expand while fossil shares contract, this provides visual evidence of decarbonization.

If renewables are split into categories, the chart also reveals:

Which technology drives growth (e.g., wind vs solar)

Diversification of renewable capacity

Offshore vs onshore wind development

In [ ]:
# ============================================================
# ENERGY TRANSITION (100% STACKED SHARES) — REGION DROPDOWN
# - Y-axis: shares sum to 100%
# - Categories: Renewables vs Nuclear vs Thermal/Fossil
# - Optional: split Renewables into solar / wind onshore / wind offshore / hydro / bio
# - Includes "France (all regions)" aggregation
# ============================================================


# ----------------------------
# Column names (adjust if needed)
# ----------------------------
df = df_energy_cleaned
region_col = "libelle_region"
time_col = "date_heure"  # should be datetime64[ns, UTC] from your preprocessing

# Core production columns (based on your list)
nuclear_col = "nucleaire"
thermal_col = "thermique"

# Renewables (prefer onshore/offshore if available; ignore "eolien" because it's object in your data)
renew_map = {
    "Solar": "solaire",
    "Wind (Onshore)": "eolien_terrestre",
    "Wind (Offshore)": "eolien_offshore",
    "Hydro": "hydraulique",
    "Bioenergy": "bioenergies"
}

# ----------------------------
# Sanity checks
# ----------------------------
assert region_col in df.columns, f"Missing '{region_col}'"
assert time_col in df.columns, f"Missing '{time_col}'"
assert nuclear_col in df.columns, f"Missing '{nuclear_col}'"
assert thermal_col in df.columns, f"Missing '{thermal_col}'"

# Make sure renewables columns exist; we keep only those present
renew_map = {k: v for k, v in renew_map.items() if v in df.columns}
assert len(renew_map) > 0, "No renewable production columns found (check column names)."

# Convert any renewable columns to numeric if needed (safe even if already float)
for col in list(renew_map.values()) + [nuclear_col, thermal_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Region options (+ France total)
regions = sorted(df[region_col].dropna().unique().tolist())
region_options = ["France (all regions)"] + regions


# ----------------------------
# Helper: build aggregated dataset (region or France total)
# ----------------------------
def get_region_df(region_choice: str) -> pd.DataFrame:
    """
    Returns a dataframe with datetime index and production columns.
    If region_choice == 'France (all regions)', sums across regions per timestamp.
    """
    if region_choice == "France (all regions)":
        d = df.copy()
        # Sum across regions per timestamp
        agg = d.groupby(time_col)[list(renew_map.values()) + [nuclear_col, thermal_col]].sum(min_count=1)
        return agg
    else:
        d = df[df[region_col] == region_choice].copy()
        # For a single region we can just set index; still aggregate over time later
        d = d.set_index(time_col)[list(renew_map.values()) + [nuclear_col, thermal_col]]
        return d


# ----------------------------
# Main plot function
# ----------------------------
def fig_energy_transition(region_choice: str, freq: str = "ME", split_renewables: bool = False):
    """
    freq: "ME" monthly end or "YE" year end
    split_renewables:
      False -> Renewables aggregated into one band
      True  -> Renewables split into sub-categories
    """
    d = get_region_df(region_choice)

    # Resample to trend frequency (mean). If you prefer sums, change .mean() -> .sum()
    trend = d.resample(freq).mean()

    # Build category columns
    trend["Nuclear"] = trend[nuclear_col]
    trend["Thermal/Fossil"] = trend[thermal_col]

    # Renewables total
    renew_cols = list(renew_map.values())
    trend["Renewables"] = trend[renew_cols].sum(axis=1)

    # Total production considered in this chart
    trend["Total"] = trend["Renewables"] + trend["Nuclear"] + trend["Thermal/Fossil"]

    # Avoid divide-by-zero and drop empty rows
    trend = trend.replace([np.inf, -np.inf], np.nan)
    trend = trend.dropna(subset=["Total"])
    trend = trend[trend["Total"] > 0]

    # Shares (%)
    if split_renewables:
        # renewable shares by component
        for label, col in renew_map.items():
            trend[label] = (trend[col] / trend["Total"]) * 100

        trend["Nuclear share"] = (trend["Nuclear"] / trend["Total"]) * 100
        trend["Thermal/Fossil share"] = (trend["Thermal/Fossil"] / trend["Total"]) * 100

        plot_cols = list(renew_map.keys()) + ["Nuclear share", "Thermal/Fossil share"]
        plot_df = trend[plot_cols].reset_index().rename(columns={time_col: "period"})

        long_df = plot_df.melt(id_vars=["period"], var_name="Category", value_name="Share (%)")

        title = f"Energy Mix Shift (100% Shares) — {region_choice} — Renewables split"
        fig = px.area(
            long_df,
            x="period", y="Share (%)", color="Category",
            title=title,
            template="plotly_white"
        )

    else:
        trend["Renewables share"] = (trend["Renewables"] / trend["Total"]) * 100
        trend["Nuclear share"] = (trend["Nuclear"] / trend["Total"]) * 100
        trend["Thermal/Fossil share"] = (trend["Thermal/Fossil"] / trend["Total"]) * 100

        plot_df = trend[["Renewables share", "Nuclear share", "Thermal/Fossil share"]].reset_index().rename(columns={time_col: "period"})
        long_df = plot_df.melt(id_vars=["period"], var_name="Category", value_name="Share (%)")

        title = f"Energy Mix Shift (100% Shares) — {region_choice}"
        fig = px.area(
            long_df,
            x="period", y="Share (%)", color="Category",
            title=title,
            template="plotly_white"
        )

    # Force 0–100% axis
    fig.update_layout(
        yaxis=dict(range=[0, 100], title="Share of total production (%)"),
        xaxis_title="Time",
        hovermode="x unified",
        legend_title="Category"
    )

    return fig


# ----------------------------
# Widgets (Region + Frequency + Split toggle)
# ----------------------------
dd_region = widgets.Dropdown(options=region_options, value=region_options[0], description="Region:")
dd_freq = widgets.Dropdown(
    options=[("Monthly", "ME"), ("Yearly", "YE")],
    value="ME",
    description="Time:"
)
toggle_split = widgets.Checkbox(value=False, description="Split renewables")

out = widgets.Output()

def update(change=None):
    with out:
        out.clear_output(wait=True)
        display(fig_energy_transition(dd_region.value, freq=dd_freq.value, split_renewables=toggle_split.value))

dd_region.observe(update, names="value")
dd_freq.observe(update, names="value")
toggle_split.observe(update, names="value")

display(widgets.HBox([dd_region, dd_freq, toggle_split]), out)
update()